# Chartistry: Milestone 2 - Trending Youtube Videos

### Imports and Data Loading

In [2]:
# Basic libraries
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Display settings
pd.set_option("display.max_columns", None)
sns.set(style="whitegrid")

In [3]:
# Load datasets
data = "../data/"
df_us = pd.read_csv(data + "USvideos.csv")
df_de = pd.read_csv(data + "DEvideos.csv")
df_fr = pd.read_csv(data + "FRvideos.csv")
df_gb = pd.read_csv(data + "GBvideos.csv")
df_in = pd.read_csv(data + "INvideos.csv")
df_jp = pd.read_csv(
    data + "JPvideos.csv",
    encoding="utf-8-sig",
    encoding_errors="replace",
    engine="python"
)
df_kr = pd.read_csv(
    data + "KRvideos.csv",
    encoding="utf-8-sig",
    encoding_errors="replace",
    engine="python"
)
df_mx = pd.read_csv(
    data + "MXvideos.csv",
    encoding="utf-8-sig",
    encoding_errors="replace",
    engine="python"
)
df_ru = pd.read_csv(
    data + "RUvideos.csv",
    encoding="utf-8-sig",
    encoding_errors="replace",
    engine="python"
)

Russia, Mexico, South Korea, and Japan contain unknown / undecodable characters. The encoding characters have been replaced by "�". Let's check how many rows are affected for each datasets

In [4]:
def count_corrupted_rows(df):
    return df.apply(
        lambda row: row.astype(str).str.contains("�").any(),
        axis=1
    ).sum()

In [5]:
print("JP:", count_corrupted_rows(df_jp))
print("KR:", count_corrupted_rows(df_kr))
print("MX:", count_corrupted_rows(df_mx))
print("RU:", count_corrupted_rows(df_ru))

JP: 10
KR: 66
MX: 3
RU: 44


### Pre-processing

Let's add the country column to the dataframes and merge them all into a single dataframe

In [6]:
# Add a country column to each dataset
df_us["country"] = "US"
df_de["country"] = "DE"
df_fr["country"] = "FR"
df_gb["country"] = "GB"
df_in["country"] = "IN"
df_jp["country"] = "JP"
df_kr["country"] = "KR"
df_mx["country"] = "MX"
df_ru["country"] = "RU"

In [7]:
# Combine all datasets into one
df = pd.concat(
    [df_us, df_de, df_fr, df_gb, df_in, df_jp, df_kr, df_mx, df_ru],
    ignore_index=True
)

# Quick check
df.shape

(335061, 17)

We inspect the data and check for missing values

In [8]:
# First rows
df.head()

,video_id,trending_date,title,channel_title,category_id,publish_time,tags,views,likes,dislikes,comment_count,thumbnail_link,comments_disabled,ratings_disabled,video_error_or_removed,description,country
0,2kyS6SvSYSE,17.14.11,WE WANT TO TALK ABOUT OUR MARRIAGE,CaseyNeistat,22,2017-11-13T17:13:01.000Z,SHANtell martin,748374,57527,2966,15954,https://i.ytimg.com/vi/2kyS6SvSYSE/default.jpg,False,False,False,SHANTELL'S CHANNEL - https://www.youtube.com/s...,US
1,1ZAPwfrtAFY,17.14.11,The Trump Presidency: Last Week Tonight with J...,LastWeekTonight,24,2017-11-13T07:30:00.000Z,"last week tonight trump presidency|""last week ...",2418783,97185,6146,12703,https://i.ytimg.com/vi/1ZAPwfrtAFY/default.jpg,False,False,False,"One year after the presidential election, John...",US
2,5qpjK5DgCt4,17.14.11,"Racist Superman | Rudy Mancuso, King Bach & Le...",Rudy Mancuso,23,2017-11-12T19:05:24.000Z,"racist superman|""rudy""|""mancuso""|""king""|""bach""...",3191434,146033,5339,8181,https://i.ytimg.com/vi/5qpjK5DgCt4/default.jpg,False,False,False,WATCH MY PREVIOUS VIDEO ▶ \n\nSUBSCRIBE ► http...,US
3,puqaWrEC7tY,17.14.11,Nickelback Lyrics: Real or Fake?,Good Mythical Morning,24,2017-11-13T11:00:04.000Z,"rhett and link|""gmm""|""good mythical morning""|""...",343168,10172,666,2146,https://i.ytimg.com/vi/puqaWrEC7tY/default.jpg,False,False,False,Today we find out if Link is a Nickelback amat...,US
4,d380meD0W0M,17.14.11,I Dare You: GOING BALD!?,nigahiga,24,2017-11-12T18:01:41.000Z,"ryan|""higa""|""higatv""|""nigahiga""|""i dare you""|""...",2095731,132235,1989,17518,https://i.ytimg.com/vi/d380meD0W0M/default.jpg,False,False,False,I know it's been a while since we did this sho...,US


In [9]:
# Dataset info
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 335061 entries, 0 to 335060
Data columns (total 17 columns):
 #   Column                  Non-Null Count   Dtype 
---  ------                  --------------   ----- 
 0   video_id                335061 non-null  object
 1   trending_date           335061 non-null  object
 2   title                   335061 non-null  object
 3   channel_title           335061 non-null  object
 4   category_id             335061 non-null  int64 
 5   publish_time            335061 non-null  object
 6   tags                    335061 non-null  object
 7   views                   335061 non-null  int64 
 8   likes                   335061 non-null  int64 
 9   dislikes                335061 non-null  int64 
 10  comment_count           335061 non-null  int64 
 11  thumbnail_link          335061 non-null  object
 12  comments_disabled       335061 non-null  bool  
 13  ratings_disabled        335061 non-null  bool  
 14  video_error_or_removed  335061 non-n

In [10]:
# Check missing values
df.isnull().sum()

video_id                      0
trending_date                 0
title                         0
channel_title                 0
category_id                   0
publish_time                  0
tags                          0
views                         0
likes                         0
dislikes                      0
comment_count                 0
thumbnail_link                0
comments_disabled             0
ratings_disabled              0
video_error_or_removed        0
description               18182
country                       0
dtype: int64

We can see that only the description column has missing values. Let's fill them with a "No description" tag, and create a new column to tell whether a video has a description or not, since it can be usefull for future analysis.

In [11]:
# Drop duplicates
df = df.drop_duplicates()

# Handle missing descriptions
df["description"] = df["description"].fillna("No description")
df["has_description"] = df["description"] != "No description"

We also convert dates to datetime, and extract / store date attributes in different columns.

In [12]:
# Convert trending_date
df["trending_date"] = df["trending_date"].astype(str).str.strip()
df["trending_date"] = pd.to_datetime(
    df["trending_date"],
    format="%y.%d.%m",
    errors="coerce"
)

# Convert publish_time
df["publish_time"] = pd.to_datetime(df["publish_time"], errors="coerce")

# Remove timezone info from publish_time so it matches trending_date
df["publish_time"] = df["publish_time"].dt.tz_localize(None)

# Extract temporal features from publish_time
df["publish_hour"] = df["publish_time"].dt.hour
df["publish_day"] = df["publish_time"].dt.day_name()
# Days between publication and trending
df["days_to_trending"] = (df["trending_date"] - df["publish_time"]).dt.days

df.head()

,video_id,trending_date,title,channel_title,category_id,publish_time,tags,views,likes,dislikes,comment_count,thumbnail_link,comments_disabled,ratings_disabled,video_error_or_removed,description,country,has_description,publish_hour,publish_day,days_to_trending
0,2kyS6SvSYSE,2017-11-14,WE WANT TO TALK ABOUT OUR MARRIAGE,CaseyNeistat,22,2017-11-13 17:13:01,SHANtell martin,748374,57527,2966,15954,https://i.ytimg.com/vi/2kyS6SvSYSE/default.jpg,False,False,False,SHANTELL'S CHANNEL - https://www.youtube.com/s...,US,True,17,Monday,0
1,1ZAPwfrtAFY,2017-11-14,The Trump Presidency: Last Week Tonight with J...,LastWeekTonight,24,2017-11-13 07:30:00,"last week tonight trump presidency|""last week ...",2418783,97185,6146,12703,https://i.ytimg.com/vi/1ZAPwfrtAFY/default.jpg,False,False,False,"One year after the presidential election, John...",US,True,7,Monday,0
2,5qpjK5DgCt4,2017-11-14,"Racist Superman | Rudy Mancuso, King Bach & Le...",Rudy Mancuso,23,2017-11-12 19:05:24,"racist superman|""rudy""|""mancuso""|""king""|""bach""...",3191434,146033,5339,8181,https://i.ytimg.com/vi/5qpjK5DgCt4/default.jpg,False,False,False,WATCH MY PREVIOUS VIDEO ▶ \n\nSUBSCRIBE ► http...,US,True,19,Sunday,1
3,puqaWrEC7tY,2017-11-14,Nickelback Lyrics: Real or Fake?,Good Mythical Morning,24,2017-11-13 11:00:04,"rhett and link|""gmm""|""good mythical morning""|""...",343168,10172,666,2146,https://i.ytimg.com/vi/puqaWrEC7tY/default.jpg,False,False,False,Today we find out if Link is a Nickelback amat...,US,True,11,Monday,0
4,d380meD0W0M,2017-11-14,I Dare You: GOING BALD!?,nigahiga,24,2017-11-12 18:01:41,"ryan|""higa""|""higatv""|""nigahiga""|""i dare you""|""...",2095731,132235,1989,17518,https://i.ytimg.com/vi/d380meD0W0M/default.jpg,False,False,False,I know it's been a while since we did this sho...,US,True,18,Sunday,1


In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 322491 entries, 0 to 335060
Data columns (total 21 columns):
 #   Column                  Non-Null Count   Dtype         
---  ------                  --------------   -----         
 0   video_id                322491 non-null  object        
 1   trending_date           322491 non-null  datetime64[ns]
 2   title                   322491 non-null  object        
 3   channel_title           322491 non-null  object        
 4   category_id             322491 non-null  int64         
 5   publish_time            322491 non-null  datetime64[ns]
 6   tags                    322491 non-null  object        
 7   views                   322491 non-null  int64         
 8   likes                   322491 non-null  int64         
 9   dislikes                322491 non-null  int64         
 10  comment_count           322491 non-null  int64         
 11  thumbnail_link          322491 non-null  object        
 12  comments_disabled       322491 non-

### Add categories

In [14]:
import json
import os

# Path to category files
data = "../data/"

# Countries you have
countries = ["US", "DE", "FR", "GB", "IN", "JP", "KR", "MX", "RU", "CA"]

category_mapping = []

for country in countries:
    path = os.path.join(data, f"{country}_category_id.json")
    
    with open(path) as f:
        data_json = json.load(f)
    
    for item in data_json["items"]:
        category_mapping.append({
            "category_id": int(item["id"]),
            "category_name": item["snippet"]["title"],
            "country": country
        })

# Convert to DataFrame
df_categories = pd.DataFrame(category_mapping)

df_categories.head()

,category_id,category_name,country
0,1,Film & Animation,US
1,2,Autos & Vehicles,US
2,10,Music,US
3,15,Pets & Animals,US
4,17,Sports,US


In [15]:
df = df.merge(
    df_categories,
    on=["category_id", "country"],
    how="left"
)

In [17]:
df.head()

,video_id,trending_date,title,channel_title,category_id,publish_time,tags,views,likes,dislikes,comment_count,thumbnail_link,comments_disabled,ratings_disabled,video_error_or_removed,description,country,has_description,publish_hour,publish_day,days_to_trending,category_name
0,2kyS6SvSYSE,2017-11-14,WE WANT TO TALK ABOUT OUR MARRIAGE,CaseyNeistat,22,2017-11-13 17:13:01,SHANtell martin,748374,57527,2966,15954,https://i.ytimg.com/vi/2kyS6SvSYSE/default.jpg,False,False,False,SHANTELL'S CHANNEL - https://www.youtube.com/s...,US,True,17,Monday,0,People & Blogs
1,1ZAPwfrtAFY,2017-11-14,The Trump Presidency: Last Week Tonight with J...,LastWeekTonight,24,2017-11-13 07:30:00,"last week tonight trump presidency|""last week ...",2418783,97185,6146,12703,https://i.ytimg.com/vi/1ZAPwfrtAFY/default.jpg,False,False,False,"One year after the presidential election, John...",US,True,7,Monday,0,Entertainment
2,5qpjK5DgCt4,2017-11-14,"Racist Superman | Rudy Mancuso, King Bach & Le...",Rudy Mancuso,23,2017-11-12 19:05:24,"racist superman|""rudy""|""mancuso""|""king""|""bach""...",3191434,146033,5339,8181,https://i.ytimg.com/vi/5qpjK5DgCt4/default.jpg,False,False,False,WATCH MY PREVIOUS VIDEO ▶ \n\nSUBSCRIBE ► http...,US,True,19,Sunday,1,Comedy
3,puqaWrEC7tY,2017-11-14,Nickelback Lyrics: Real or Fake?,Good Mythical Morning,24,2017-11-13 11:00:04,"rhett and link|""gmm""|""good mythical morning""|""...",343168,10172,666,2146,https://i.ytimg.com/vi/puqaWrEC7tY/default.jpg,False,False,False,Today we find out if Link is a Nickelback amat...,US,True,11,Monday,0,Entertainment
4,d380meD0W0M,2017-11-14,I Dare You: GOING BALD!?,nigahiga,24,2017-11-12 18:01:41,"ryan|""higa""|""higatv""|""nigahiga""|""i dare you""|""...",2095731,132235,1989,17518,https://i.ytimg.com/vi/d380meD0W0M/default.jpg,False,False,False,I know it's been a while since we did this sho...,US,True,18,Sunday,1,Entertainment


In [18]:
# Select useful columns
columns = [
    "video_id",
    "title",
    "channel_title",
    "category_id",
    "category_name",
    "country",
    "views",
    "likes",
    "dislikes",
    "comment_count",
    "publish_hour",
    "publish_day",
    "days_to_trending",
    "thumbnail_link"
]

df_final = df[columns]

# Save CSV
df_final.to_csv("../data/data_final.csv", index=False)

In [19]:
test = pd.read_csv("../data/data_final.csv")